![Piggy bank](piggy_bank.jpg)

Personal loans are a lucrative revenue stream for banks. The typical interest rate of a two-year loan in the United Kingdom is [around 10%](https://www.experian.com/blogs/ask-experian/whats-a-good-interest-rate-for-a-personal-loan/). This might not sound like a lot, but in September 2022 alone UK consumers borrowed [around £1.5 billion](https://www.ukfinance.org.uk/system/files/2022-12/Household%20Finance%20Review%202022%20Q3-%20Final.pdf), which would mean approximately £300 million in interest generated by banks over two years!

You have been asked to work with a bank to clean the data they collected as part of a recent marketing campaign, which aimed to get customers to take out a personal loan. They plan to conduct more marketing campaigns going forward so would like you to ensure it conforms to the specific structure and data types that they specify so that they can then use the cleaned data you provide to set up a PostgreSQL database, which will store this campaign's data and allow data from future campaigns to be easily imported.

## `client.csv`

| column | data type | description | cleaning requirements |
|--------|-----------|-------------|-----------------------|
| `client_id` | `integer` | Client ID | N/A |
| `age` | `integer` | Client's age in years | N/A |
| `job` | `object` | Client's type of job | Change `"."` to `"_"` |
| `marital` | `object` | Client's marital status | N/A |
| `education` | `object` | Client's level of education | Change `"."` to `"_"` and `"unknown"` to `np.NaN` |
| `credit_default` | `bool` | Whether the client's credit is in default | Convert to `boolean` data type:<br> `1` if `"yes"`, otherwise `0` |
| `mortgage` | `bool` | Whether the client has an existing mortgage (housing loan) | Convert to boolean data type:<br> `1` if `"yes"`, otherwise `0` |

<br>

## `campaign.csv`

| column | data type | description | cleaning requirements |
|--------|-----------|-------------|-----------------------|
| `client_id` | `integer` | Client ID | N/A |
| `number_contacts` | `integer` | Number of contact attempts to the client in the current campaign | N/A |
| `contact_duration` | `integer` | Last contact duration in seconds | N/A |
| `previous_campaign_contacts` | `integer` | Number of contact attempts to the client in the previous campaign | N/A |
| `previous_outcome` | `bool` | Outcome of the previous campaign | Convert to boolean data type:<br> `1` if `"success"`, otherwise `0`. |
| `campaign_outcome` | `bool` | Outcome of the current campaign | Convert to boolean data type:<br> `1` if `"yes"`, otherwise `0`. |
| `last_contact_date` | `datetime` | Last date the client was contacted | Create from a combination of `day`, `month`, and a newly created `year` column (which should have a value of `2022`); <br> **Format =** `"YYYY-MM-DD"` |

<br>

## `economics.csv`

| column | data type | description | cleaning requirements |
|--------|-----------|-------------|-----------------------|
| `client_id` | `integer` | Client ID | N/A |
| `cons_price_idx` | `float` | Consumer price index (monthly indicator) | N/A |
| `euribor_three_months` | `float` | Euro Interbank Offered Rate (euribor) three-month rate (daily indicator) | N/A |

In [79]:
import pandas as pd
import numpy as np

# Start coding here...

In [80]:
df = pd.read_csv("bank_marketing.csv")

for col in ["credit_default", "mortgage", "previous_outcome", "campaign_outcome"]:
    print(col)
    print("--------------")
    print(df[col].value_counts())

credit_default
--------------
no         32588
unknown     8597
yes            3
Name: credit_default, dtype: int64
mortgage
--------------
yes        21576
no         18622
unknown      990
Name: mortgage, dtype: int64
previous_outcome
--------------
nonexistent    35563
failure         4252
success         1373
Name: previous_outcome, dtype: int64
campaign_outcome
--------------
no     36548
yes     4640
Name: campaign_outcome, dtype: int64


In [81]:
df = pd.read_csv("bank_marketing.csv")
data = pd.DataFrame(df)
print(data.dtypes)

client_id                       int64
age                             int64
job                            object
marital                        object
education                      object
credit_default                 object
mortgage                       object
month                          object
day                             int64
contact_duration                int64
number_contacts                 int64
previous_campaign_contacts      int64
previous_outcome               object
cons_price_idx                float64
euribor_three_months          float64
campaign_outcome               object
dtype: object


In [82]:
#client.csv file
new_data = data[['client_id', 'age', 'job', 'marital', 'education', 'credit_default', 'mortgage']]
new_data['job'] = new_data['job'].str.replace('.', '_')
new_data['education'] = new_data['education'].str.replace('.', '_') 
new_data['education']=new_data['education'].replace('unknown', np.nan)

for i in range(new_data.shape[0]):
    if new_data.iloc[i,5] == 'yes':
        new_data.iloc[i,5] = 1
    else:
        new_data.iloc[i,5] = 0
        
for i in range(new_data.shape[0]):
    if new_data.iloc[i,6] == 'yes':
        new_data.iloc[i,6] = 1
    else:
        new_data.iloc[i,6] = 0
new_data['credit_default'] = new_data['credit_default'].astype(bool)
new_data['mortgage'] = new_data['mortgage'].astype(bool)
new_data.to_csv('client.csv', index=False)

In [83]:
#campaign.csv file

old = data[['client_id', 'number_contacts', 'contact_duration', 'previous_campaign_contacts', 'previous_outcome', 'campaign_outcome', 'month', 'day']]
old['month'] = pd.to_datetime(old['month'], format='%b').dt.month
old['last_contact_date'] = pd.to_datetime(old[['month', 'day']].assign(year=2022))
main = old[['client_id', 'number_contacts', 'contact_duration', 'previous_campaign_contacts', 'previous_outcome', 'campaign_outcome', 'last_contact_date']]

for i in range(main.shape[0]):
    if main.iloc[i,4] == 'success':
        main.iloc[i,4] = 1
    else:
        main.iloc[i,4] = 0

for i in range(main.shape[0]):
    if main.iloc[i,5] == 'yes':
        main.iloc[i,5] = 1
    else:
        main.iloc[i,5] = 0
main['previous_outcome'] = main['previous_outcome'].astype(bool)
main['campaign_outcome'] = main['campaign_outcome'].astype(bool)
main.to_csv('campaign.csv', index=False)

In [84]:
#economics.csv file

econ = data[['client_id', 'cons_price_idx', 'euribor_three_months']]
econ.to_csv('economics.csv', index=False)